# Exploring PDE constrained optimisation

This notebook will document my process of exploring PDE constrained optimisation through mathematics and implementation in pure Python.

## An initial problem

Let's look at a Poisson optimal control problem. 

The Poisson equation is one of the most fundamental PDEs in matehmatical physics. It reads

$$-\nabla^2\varphi = q,$$

where $\varphi$ and $q$ are real or complex valued functions. We use the minus sign here to ensure $\nabla^2$ is positive definite, which guarantees a unique solution. Explictly, in 3D Cartesian space this looks like

$$-\left(\frac{\partial^2}{\partial x^2} + \frac{\partial^2}{\partial y^2} + \frac{\partial^2}{\partial z^2}\right)\varphi(x, y, z) = q(x, y, z).$$

Poisson's equation has applications across gravity, fluid dynamics, thermodynamics, and more. 

In this notebook, we will consider that $\varphi$ is a 2D steady state temperature field and $q$ is a heat source/ sink distribution (e.g., $q > 0$ at a point means heat injection, $q < 0$ means heat absoprtion). For some domain $\Omega \subset \mathbb{R}^{2}$ with boundary $\delta\Omega$, we will say that we want our temperature to be zero at the boundaries, i.e., our problem in full is

$$-\nabla^2 \varphi = q \in \Omega,\;\;\;\; \varphi = 0 \text{ on } \delta\Omega.$$

### PDE constrained optimiation

PDE constrained optimisation problems usually take the form

$$\text{minimise } J(u, m) \text{  subject to  } g(u, m) = 0,$$

where $J$ is the functional, $g$ is the constraint expressing our PDE and boundary conditions, and $u$ and $m$ are model parameters.  

In the case that our constraint is a PDE (and associated initial and boundary conditions), our workflow might be

$$\underbrace{m}_\text{input} \;\; \rightarrow \;\; \underbrace{g(u, m)}_\text{PDE constraint} \;\; \rightarrow \;\; \underbrace{u}_\text{output}\;\; \rightarrow \underbrace{J(u, m)}_\text{functional}.$$

In this case our objective is to find $\frac {\partial J} {\partial m}.$ We want to figure out how our objective varies with respect to particular model parameters in order to update those parameters.

In the context of our problem, the optimal control problem is such that we are allowed to choose $f$ and we want to choose it so that the resulting temperature $\varphi$ is as close as possible to some desired target profile $\hat{\varphi}$. In essence, we want to minimise the difference between a given and target temperature profile subject to the Poisson equation (and specified boundary conditions).

### An intuitive formulation

Let's make this formulation concrete and interpretable.

Consider a top down view of a square room with side lengths of $1m$. Let's say it is freezing outside, such that the temperature of the walls is $0$. Inside the room there are sources and sinks (radiators, or people), each with a fixed heat output magnitude $Q$. They radiate or cool, and $f$ describes their distribution in the room. The question is: where should we place these to make the room temperature $\varphi$ as close to the desired profile $\hat{\varphi}$ as possible?

More formally, consider a domain $\Omega = [0, 1]^2$ subject to the boundary conditions

$$\varphi(x, 0) = 0\\ \varphi(x, 1) = 0 \\ \varphi(0, y) = 0 \\ \varphi(1, y) = 0.$$

If we let $q$ represent our heat source/sink distribution, our problem then is 

$$\text{minimise } J(\varphi, q) = \frac 1 2 \int_\Omega (\varphi - \hat{\varphi})^2\ dx dy + \frac \beta 2 \int_\Omega q^2\ dxdy \\
\text{subject to  } g(\varphi, f) = \begin{cases}
-\nabla^2 \varphi - q = 0\\
\varphi = 0 \text{ on } \delta \Omega
\end{cases}.$$

In the functional, the first term measures how well the temperature matches the target profile, while the second penalises excessive heating or cooling. Here, $\beta$ is a regularisation parameter which punishes large field values.

For simplicity, we will treat $f$ as a continuous field across the whole room that represents source/sink placement. Rather than optimising placement directly, the optimiser will be allowed to distribute heating and cooling across the whole room and positioning will be implied by regions that are strongly positive/negative. This is for simplicity and mathematical convenience.

### Discretisation

Notice the above formulation is continuous. To solve this problem on a finite machine, we must of course discretise our problem. Consider our domain is broken down into $N + 2$ points in each direction (+2 ghost nodes to handle boundary conditions). Then our grid spacing is

$$x_i = ih,\;\;\;\; y_j = jh,$$

$i, j = 0, 1, \dots, N$, $h = 1/N+1$.

Now, our PDE is

$$-\nabla^2\varphi(x, y) - q(x, y) = 0$$

Since our domain is uniform and a simple shape, we can just use finite differences to discretise our equation. We won't concern ourselves with the implementation here, but just know the Laplacian can be written in discrete form using second order central differences as

$$\nabla^2 \varphi(x_i, y_j) = \frac{\varphi_{i+1, j} + \varphi_{i-1, j} + \varphi_{i, j+1} + \varphi_{i, j-1} - 4\varphi_{i, j}}{h^2},$$

where we have used the notation that $\varphi(x+h, y) := \varphi_{i+1, j}$ for convience. Now, we can write our PDE in discrete form as

$$-\frac{\varphi_{i+1, j} + \varphi_{i-1, j} + \varphi_{i, j+1} + \varphi_{i, j-1} - 4\varphi_{i, j}}{h^2} - q_{i, j} = 0.$$

Note that we can write this problem in the form 

$$A\varphi - q = 0,$$

where $A$ is a (block) tridiagonal discretisation matrix (with tridiagonal blocks) of the form

$$
A = \frac{1}{h^2}
\begin{bmatrix}
B & -I & 0 & \cdots & 0 \\
-I & B & -I & \ddots & \vdots \\
0 & -I & B & \ddots & 0 \\
\vdots & \ddots & \ddots & \ddots & -I \\
0 & \cdots & 0 & -I & B
\end{bmatrix},
$$

where $I$ is the $N \times N$ identity matrix and $B$ is the tridiagonal block

$$
B =
\begin{bmatrix}
4 & -1 & 0 & \cdots & 0 \\
-1 & 4 & -1 & \ddots & \vdots \\
0 & -1 & 4 & \ddots & 0 \\
\vdots & \ddots & \ddots & \ddots & -1 \\
0 & \cdots & 0 & -1 & 4
\end{bmatrix}.
$$

We can use a similar approach to express our boundary conditions, which become

$$\varphi_{0, j} = \varphi_{N+1, j} = \varphi_{i, 0} = \varphi_{i, N+1} = 0.$$

Since we are considering $N+2$ points, our continuous integrals in the functional simply become discrete summations, and our fully discretised optimisation problem becomes

$$
\begin{align*}
    \text{minimise } & J(\varphi, q) = \frac {h^2} 2 \sum_{i, j} (\varphi_{i, j} - \hat{\varphi_{i, j}})^2 + \frac {h^2\beta} 2 \sum_{i, j} q_{i, j}^2 \\[5pt]
    
    \text{subject to  } & g(\varphi, f) = 
    
    \begin{cases}
        A\varphi_{i+1, j} - q_{i, j} = 0\\
        \varphi_{0, j} = 0\\ 
        \varphi_{N+1, j} =0 \\
        \varphi_{i, 0} = 0\\
        \varphi_{i, N+1} = 0
    \end{cases}.
\end{align*}$$

## The reduced problem

## The adjoint equation

## The gradient of the reduced functional

## Discretising everything

## Implementation

1. Initial guess
2. Solve state system $A\varphi^{(k)} = q^{(k)}$
3. Solve the adjoint system $Ap^{(k)} = \varphi^{(k)} - \hat{\varphi}$
4. Form the gradient $g^{(k)} = \beta q^{(k)} - p^{(k)}$
5. Update $q$ using gradient descent, conjugate gradient, or L-BFGS